# PRACTICE 1: CLASSIFYING SPAM EMAILS
## NHÓM 3 — ĐẠI HỌC GIAO THÔNG VẬN TẢI (UTH)
> **Giảng viên hướng dẫn:** Tiến sĩ Nguyễn Thị Khánh Tiên  
> **Đề tài:** Phân loại email spam/ham bằng học máy (Naive Bayes, Logistic Regression, Linear SVM)  
> **Tập dữ liệu nguồn:** `data/raw/CEAS_08.csv` (39.154 email thực tế từ hội thảo CEAS)

```text
[CEAS_08.csv] ──> TV 1 (EDA Nền tảng: Cấu trúc, Chất lượng & Đơn biến)
              ──> TV 2 (EDA Chuyên sâu: Đa biến, Ngoại lai & Đúc kết định hướng)
              ──> TV 3 (Làm sạch dữ liệu: Khử trùng, điền missing, clean_text)
              ──> TV 4 (FE Biến đổi: Log, Binning, Tạo biến Domain, Mã hóa)
              ──> TV 5 (FE Lọc & Pipeline: Chia Train/Test chống Leak, TF-IDF, Chi2)
              ──> TV 6 (Huấn luyện 3 mô hình: Naive Bayes, Logistic L1, Linear SVM)
              ──> TV 7 (Đánh giá kiểm thử, Phân tích lỗi & Web Demo Streamlit)
```

---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)
**Phụ trách:** Nguyễn Xuân Đông (Trưởng nhóm)


In [ ]:
# ==============================================================================
# 1.1 KHAI BÁO THƯ VIỆN & CẤU HÌNH RANDOM SEED
# ==============================================================================
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
import re
import json
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

# Cố định seed ngẫu nhiên cho toàn bộ quy trình
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

DATA_RAW_PATH = '../data/raw/CEAS_08.csv'

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
print(f'✅ Môi trường sẵn sàng với Python {sys.version.split()[0]} | RANDOM_SEED = {RANDOM_SEED}')

In [ ]:
# ==============================================================================
# 1.2 KHẢO SÁT CẤU TRÚC & ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU (EDA BƯỚC 1 & 2)
# ==============================================================================
df = pd.read_csv(DATA_RAW_PATH)

print(f'📊 Tổng số mẫu (email): {df.shape[0]:,} dòng')
print(f'📊 Tổng số biến: {df.shape[1]} cột\n')

print('--- [1. CẤU TRÚC] THÔNG TIN ĐỊNH DẠNG & KIỂU DỮ LIỆU ---')
display(df.info())

print('\n--- 5 MẪU DỮ LIỆU ĐẦU TIÊN ---')
display(df.head(5))

print('\n--- [2. CHẤT LƯỢNG] THỐNG KÊ GIÁ TRỊ THIẾU (MISSING VALUES) ---')
missing_series = df.isnull().sum()
missing_percent = (missing_series / len(df)) * 100
missing_summary = pd.DataFrame({'Số lượng thiếu': missing_series, 'Tỷ lệ thiếu (%)': missing_percent})
display(missing_summary[missing_summary['Số lượng thiếu'] > 0])

dup_count = df.duplicated().sum()
print(f'\n--- [2. CHẤT LƯỢNG] THỐNG KÊ DÒNG TRÙNG LẶP: {dup_count:,} dòng ({(dup_count/len(df))*100:.2f}%) ---')


In [ ]:
# ==============================================================================
# 1.3 PHÂN TÍCH ĐƠN BIẾN TOÀN DIỆN (UNIVARIATE ANALYSIS - EDA BƯỚC 3)
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# 1. Đơn biến mục tiêu: Phân phối nhãn (label)
label_counts = df['label'].value_counts()
sns.countplot(data=df, x='label', palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title(f'1. Biến mục tiêu: 0 (Ham: {label_counts[0]/len(df)*100:.1f}%) vs 1 (Spam: {label_counts[1]/len(df)*100:.1f}%)')
axes[0, 0].set_xlabel('Nhãn (0: Ham, 1: Spam)')
axes[0, 0].set_ylabel('Số lượng email')

# 2. Đơn biến số: Phân bố có/không có link URL (urls)
sns.countplot(data=df, x='urls', palette='viridis', ax=axes[0, 1])
url_rate = (df['urls'].sum() / len(df)) * 100
axes[0, 1].set_title(f'2. Biến số URLs: {url_rate:.1f}% email có chứa link URL')
axes[0, 1].set_xlabel('Chứa liên kết URL (0: Không, 1: Có)')
axes[0, 1].set_ylabel('Số lượng email')

# 3. Đơn biến văn bản: Phân phối độ dài tiêu đề (subject length)
subj_lengths = df['subject'].dropna().astype(str).apply(len)
sns.histplot(subj_lengths, bins=40, kde=True, color='teal', ax=axes[1, 0])
axes[1, 0].set_title(f'3. Phân phối độ dài tiêu đề Subject (Trung bình: {subj_lengths.mean():.1f} ký tự)')
axes[1, 0].set_xlabel('Số ký tự trong tiêu đề')
axes[1, 0].set_ylabel('Tần số')

# 4. Đơn biến danh mục: Top 10 tên miền người gửi (Sender Domain)
def get_domain(s):
    m = re.search(r'@([\w\.-]+)', str(s))
    return m.group(1).lower() if m else 'unknown'
top_domains = df['sender'].apply(get_domain).value_counts().head(10)
sns.barplot(x=top_domains.values, y=top_domains.index, palette='mako', ax=axes[1, 1])
axes[1, 1].set_title('4. Top 10 Tên miền người gửi phổ biến nhất')
axes[1, 1].set_xlabel('Số lượng email')

plt.tight_layout()
plt.show()


---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 2: Viết code phân tích đa biến, vẽ boxplot theo nhãn, tính ngoại lai IQR, Correlation Heatmap và đúc kết kết luận EDA tại đây


---
## Mục 3: Tiền xử lý dữ liệu & Làm sạch văn bản (Data Cleaning)
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 3: Viết code khử trùng lặp, điền missing subject, gộp text và hàm clean_text (HTML, regex, stopwords) tại đây


---
## Mục 4: Feature Engineering I — Biến đổi đặc trưng & Tạo biến Domain
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 4: Viết code Log-transform, Binning độ dài, mã hóa Ordinal và tạo đặc trưng domain (caps_ratio, num_excl, num_dollar) tại đây


---
## Mục 5: Feature Engineering II — Trích xuất TF-IDF, Chọn lọc đặc trưng & Pipeline
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 5: Viết code chia Train/Test chống Leakage, trích xuất TF-IDF, lọc Chi-square và đóng gói Pipeline tại đây


---
## Mục 6: Huấn luyện mô hình — Naive Bayes, Logistic Regression & Linear SVM
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 6: Viết code huấn luyện Naive Bayes, Logistic Regression (L1) và Linear SVM (Platt Scaling) với 5-Fold CV tại đây


---
## Mục 7: Đánh giá so sánh 3 mô hình, Phân tích lỗi & Triển khai Demo
**Phụ trách:** [Chưa chỉ định]


In [ ]:
# TV 7: Viết code đánh giá trên Test, vẽ Confusion Matrix, Error Analysis, Threshold Tuning và xuất Best Model tại đây
